# Cirrhosis Stage Prediction - Baseline Models

**Objective**: Establish baseline performance using classical ML algorithms before synthetic data augmentation. This notebook implements a leak-free modeling pipeline with proper cross-validation.

**Author**: Michael Udousoro  
**Date**: January 15, 2026

**Key Features**:
- Stratified 5-fold cross-validation to handle class imbalance
- Leak-free preprocessing (imputation only on training folds)
- KNN imputation sensitivity analysis

In [30]:
import numpy as np
import pandas as pd
import random

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

DATA_PATH = "../data/raw/cirrhosis.csv"
TARGET = "Stage"

print(f"Configuration set:")
print(f"- Random seed: {SEED}")
print(f"- Target variable: {TARGET}")

Configuration set:
- Random seed: 42
- Target variable: Stage


## 1. Data Loading

Loading the cirrhosis dataset and standardizing missing value representations. The dataset contains clinical measurements from liver cirrhosis patients with the goal of predicting disease stage (1-4).

**Missing Data Handling**: Converting all missing indicators (NA, nan, None, empty strings) to `np.nan` for consistent processing.

In [31]:
df = pd.read_csv(DATA_PATH)

# Normalize missing indicators
df = df.replace(["NA", "Na", "nan", "None", ""], np.nan)

# Drop ID if present
if "ID" in df.columns:
    df = df.drop(columns=["ID"])

print(f"Dataset loaded: {df.shape}")
print(f"\nFirst few rows:")
df.head()

Dataset loaded: (418, 19)

First few rows:


,N_Days,Status,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,400,D,D-penicillamine,21464,F,Y,Y,Y,Y,14.5,261.0,2.60,156.0,1718.0,137.95,172.0,190.0,12.2,4.0
1,4500,C,D-penicillamine,20617,F,N,Y,Y,N,1.1,302.0,4.14,54.0,7394.8,113.52,88.0,221.0,10.6,3.0
2,1012,D,D-penicillamine,25594,M,N,N,N,S,1.4,176.0,3.48,210.0,516.0,96.10,55.0,151.0,12.0,4.0
3,1925,D,D-penicillamine,19994,F,N,Y,Y,S,1.8,244.0,2.54,64.0,6121.8,60.63,92.0,183.0,10.3,4.0
4,1504,CL,Placebo,13918,F,N,Y,Y,N,3.4,279.0,3.53,143.0,671.0,113.15,72.0,136.0,10.9,3.0


## 2. Feature Engineering and Target Definition

**Critical Step**: Removing samples with missing target values before any analysis to prevent data leakage. We cannot train on samples where the outcome is unknown.

This cell:
1. Drops rows with missing Stage values (cannot be imputed)
2. Separates features (X) and target (y) 
3. Reports class distribution to assess imbalance
4. Identifies features with missing values that will be imputed

In [32]:
# Drop rows where target is missing (cannot train on these)
df = df.dropna(subset=[TARGET]).copy()

print(f"After dropping missing targets: {df.shape[0]} samples")

y = df[TARGET]
X = df.drop(columns=[TARGET])

print(f"\nFeatures: {X.shape[1]} columns, {X.shape[0]} samples")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nMissing values per feature:")
missing = X.isnull().sum()[X.isnull().sum() > 0]
if len(missing) > 0:
    print(missing)
else:
    print("No missing values in features!")

After dropping missing targets: 412 samples

Features: 18 columns, 412 samples

Target distribution:
Stage
3.0    155
4.0    144
2.0     92
1.0     21
Name: count, dtype: int64

Missing values per feature:
Drug             100
Ascites          100
Hepatomegaly     100
Spiders          100
Cholesterol      128
Copper           102
Alk_Phos         100
SGOT             100
Tryglicerides    130
Platelets         11
Prothrombin        2
dtype: int64


## 3. Preprocessing Pipeline (Leak-Free Design)

**Key Innovation**: Using scikit-learn pipelines ensures preprocessing (imputation, scaling, encoding) fits only on training data during cross-validation, preventing data leakage.

**Numeric Features**:
- KNN imputation (k=5): Preserves feature relationships better than mean/median
- StandardScaler: Required for distance-based algorithms and regularization

**Categorical Features**:
- Mode imputation: Simple, effective for categorical missingness
- One-hot encoding: Converts categories to binary features

**Why this matters**: If we imputed on the full dataset before splitting, test fold information would leak into training through the imputation process.

In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer

# Identify feature types
num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object"]).columns.tolist()

print(f"Numeric features ({len(num_features)}): {num_features}")
print(f"Categorical features ({len(cat_features)}): {cat_features}")

numeric_transformer = Pipeline(steps=[
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features),
    ],
    remainder="drop"
)

print("\nPreprocessing pipeline created!")

Numeric features (11): ['N_Days', 'Age', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin']
Categorical features (7): ['Status', 'Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']

Preprocessing pipeline created!


## 4. Baseline Model Selection

**Logistic Regression**: 
- Linear baseline with L2 regularization
- Interpretable coefficients for feature importance
- Fast training, suitable for small datasets

**Random Forest**: 
- Non-linear ensemble of 500 decision trees
- Handles feature interactions automatically
- Robust to outliers and mixed data types

These models represent fundamentally different approaches (linear vs. non-linear) and provide performance bounds before attempting synthetic augmentation.

In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Define models
logreg = LogisticRegression(max_iter=3000, random_state=SEED)
rf = RandomForestClassifier(n_estimators=500, random_state=SEED)

# Create full pipelines (preprocessing + model)
logreg_model = Pipeline(steps=[("preprocess", preprocess), ("model", logreg)])
rf_model = Pipeline(steps=[("preprocess", preprocess), ("model", rf)])

print("✓ Models created:")
print("  - Logistic Regression")
print("  - Random Forest (500 trees)")

✓ Models created:
  - Logistic Regression
  - Random Forest (500 trees)


## 5. Evaluation Protocol

**Stratified K-Fold Cross-Validation** (k=5):
- Each fold maintains the same class distribution as the full dataset
- Critical for imbalanced data (Stage 1 has only 21 samples)
- Provides robust performance estimates with confidence intervals

**Metrics**:
- **Accuracy**: Overall correctness, but can be misleading with class imbalance
- **Macro F1**: Averages F1 across all classes equally, better for imbalanced data

**Why 5 folds?**: Balances bias-variance tradeoff with computational cost. Each test fold has ~82 samples.

In [35]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score

# 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Metrics to track
scoring = {
    "acc": "accuracy",
    "f1_macro": make_scorer(f1_score, average="macro")
}

def evaluate(name, model):
    """Evaluate model using stratified cross-validation"""
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, return_train_score=False)
    
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"Accuracy:  {scores['test_acc'].mean():.3f} ± {scores['test_acc'].std():.3f}")
    print(f"F1-macro:  {scores['test_f1_macro'].mean():.3f} ± {scores['test_f1_macro'].std():.3f}")
    
    return scores

print("✓ Evaluation function ready!")

✓ Evaluation function ready!


## 6. Baseline Experiment Results

Executing cross-validation for both models. Each model trains 5 times (one per fold) with different train/test splits.

**Expected runtime**: 1-2 minutes for logistic regression, 3-5 minutes for random forest.

In [39]:
# Evaluate both models
logreg_scores = evaluate("Logistic Regression", logreg_model)
rf_scores = evaluate("Random Forest", rf_model)

print("\n" + "="*50)
print("BASELINE RESULTS COMPLETE ")
print("="*50)

/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ we


Logistic Regression
Accuracy:  0.495 ± 0.054
F1-macro:  0.352 ± 0.044


/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/util


Random Forest
Accuracy:  0.517 ± 0.042
F1-macro:  0.356 ± 0.032

BASELINE RESULTS COMPLETE 


/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


## 7. Results Summary

Consolidated table for manuscript reporting. Note the standard deviations indicate stability across different train/test splits.

**Interpretation Guide**:
- Accuracy ~50% suggests significant room for improvement (random baseline for 4 classes = 25%)
- Low F1 scores indicate difficulty predicting minority classes (especially Stage 1)
- RF slightly outperforms LogReg, suggesting non-linear patterns exist

In [37]:
# Create results summary
baseline_summary = []

for name, model in [("Logistic Regression", logreg_model), ("Random Forest", rf_model)]:
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, return_train_score=False)
    baseline_summary.append({
        "Model": name,
        "Accuracy_mean": scores["test_acc"].mean(),
        "Accuracy_std": scores["test_acc"].std(),
        "F1_macro_mean": scores["test_f1_macro"].mean(),
        "F1_macro_std": scores["test_f1_macro"].std(),
    })

baseline_df = pd.DataFrame(baseline_summary)
print("\nBaseline Results Summary:")
print(baseline_df.to_string(index=False))

/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ we


Baseline Results Summary:
              Model  Accuracy_mean  Accuracy_std  F1_macro_mean  F1_macro_std
Logistic Regression       0.495269      0.054451       0.351627      0.043983
      Random Forest       0.517103      0.042408       0.356259      0.031692


/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


## 8. KNN Imputation Sensitivity Analysis

Testing k ∈ {3, 5, 10} to justify our choice of k=5. Smaller k values use more local information but may be sensitive to noise. Larger k values are more stable but may over-smooth.

**Hypothesis**: k=5 should provide a good balance between local feature relationships and robustness.

In [41]:
print("Running sensitivity analysis for KNN imputation...\n")

for k in [3, 5, 10]:
    # Create pipeline with different k
    numeric_transformer_k = Pipeline(steps=[
        ("imputer", KNNImputer(n_neighbors=k)),
        ("scaler", StandardScaler())
    ])
    
    preprocess_k = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer_k, num_features),
            ("cat", categorical_transformer, cat_features),
        ]
    )
    
    model_k = Pipeline(steps=[
        ("preprocess", preprocess_k),
        ("model", LogisticRegression(max_iter=3000, random_state=SEED))
    ])
    
    print(f"\nLogistic Regression with KNN neighbors={k}")
    evaluate(f"LogReg (KNN={k})", model_k)

print("\n" + "="*50)
print("SENSITIVITY ANALYSIS COMPLETE ")
print("="*50)

Running sensitivity analysis for KNN imputation...


Logistic Regression with KNN neighbors=3

LogReg (KNN=3)
Accuracy:  0.485 ± 0.046
F1-macro:  0.346 ± 0.040

Logistic Regression with KNN neighbors=5

LogReg (KNN=5)
Accuracy:  0.495 ± 0.054
F1-macro:  0.352 ± 0.044

Logistic Regression with KNN neighbors=10

LogReg (KNN=10)
Accuracy:  0.498 ± 0.056
F1-macro:  0.355 ± 0.043

SENSITIVITY ANALYSIS COMPLETE 


/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ we